In [ ]:
import pandas as pd
import numpy as np

# =============================================================================
# AÑOS A PROCESAR
# =============================================================================
ANIOS = [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015,
         2016, 2017, 2018, 2019, 2020]

# =============================================================================
# FUNCIÓN AUXILIAR: agrupar edad
# =============================================================================
def agrupar_edad(edad):
    if pd.isna(edad): return '00'
    edad = int(edad)
    if edad < 16: return '00'
    elif edad <= 19: return '15'
    elif edad <= 24: return '20'
    elif edad <= 29: return '25'
    elif edad <= 34: return '30'
    elif edad <= 39: return '35'
    elif edad <= 44: return '40'
    elif edad <= 49: return '45'
    elif edad <= 54: return '50'
    elif edad <= 59: return '55'
    elif edad <= 64: return '60'
    else: return '65'

# =============================================================================
# FUNCIÓN PRINCIPAL: procesar un año
# =============================================================================
def procesar_anio(anio):
    print(f"\n{'='*60}")
    print(f"PROCESANDO AÑO {anio}")
    print(f"{'='*60}")

    # 1. Cargar EVR
    print(f"1. Cargando EVR_{anio}.csv...")
    df_evr = pd.read_csv(f'EVR_{anio}.csv', sep='\t', low_memory=False)
    print(f"   Total registros EVR: {len(df_evr):,}")

    # 2. Filtrar emigrantes (PROVALTA = 66 = extranjero)
    df_evr['PROVALTA'] = df_evr['PROVALTA'].astype(str).str.zfill(2)
    df_emigrantes = df_evr[df_evr['PROVALTA'] == '66'].copy()
    print(f"   Emigrantes al extranjero: {len(df_emigrantes):,}")

    # 3. Crear llaves para matching
    df_emigrantes['LLAVE_EDAD'] = df_emigrantes['EDAD'].apply(agrupar_edad)
    df_emigrantes['LLAVE_NAC'] = np.where(df_emigrantes['CNAC'].astype(str) == '108', '1', '2')

    # 4. Cargar EPA T4
    print(f"2. Cargando EPA_{anio}T4.csv...")
    df_epa = pd.read_csv(f'EPA_{anio}T4.csv', sep='\t')
    col_peso = df_epa.columns[-1]

    # 5. Crear matriz probabilística
    df_epa['LLAVE_EDAD'] = df_epa['EDAD5'].astype(str).str.zfill(2)
    df_epa['LLAVE_NAC'] = np.where(df_epa['NAC1'].astype(str) == '1', '1', '2')

    columnas_clave = ['LLAVE_NAC', 'LLAVE_EDAD']
    df_epa_valida = df_epa[df_epa['NFORMA'].notna()].copy()

    epa_agrupada = df_epa_valida.groupby(columnas_clave + ['NFORMA'])[col_peso].sum().reset_index()
    epa_totales = epa_agrupada.groupby(columnas_clave)[col_peso].sum().reset_index()
    epa_totales.rename(columns={col_peso: 'total'}, inplace=True)

    epa_probs = pd.merge(epa_agrupada, epa_totales, on=columnas_clave)
    epa_probs['prob'] = epa_probs[col_peso] / epa_probs['total']
    epa_matriz = epa_probs.pivot(index=columnas_clave, columns='NFORMA', values='prob').fillna(0).reset_index()

    # 6. Statistical matching
    print(f"3. Realizando statistical matching...")
    df_final = pd.merge(df_emigrantes, epa_matriz, on=columnas_clave, how='inner')

    niveles = [col for col in epa_matriz.columns if col not in columnas_clave]

    def asignar_estudios(fila):
        probs = [fila[n] for n in niveles]
        if sum(probs) == 0: return np.nan
        probs = probs / np.sum(probs)
        return np.random.choice(niveles, p=probs)

    df_final['NFORMA_IMPUTADO'] = df_final.apply(asignar_estudios, axis=1)
    df_final['ANIO'] = anio

    # Limpiamos columnas temporales
    df_final.drop(columns=niveles + ['LLAVE_EDAD', 'LLAVE_NAC'], inplace=True)

    print(f"4. Año {anio} completado: {len(df_final):,} emigrantes")

    return df_final

# =============================================================================
# EJECUTAR PARA TODOS LOS AÑOS
# =============================================================================
np.random.seed(42)

dfs_todos = []
for anio in ANIOS:
    try:
        df = procesar_anio(anio)
        dfs_todos.append(df)
    except FileNotFoundError as e:
        print(f"\n¡AVISO! Falta archivo del año {anio}: {e}")
        continue
    except Exception as e:
        print(f"\n¡ERROR en año {anio}!: {e}")
        continue

# Concatenamos todos los años en un solo DataFrame
df_completo = pd.concat(dfs_todos, ignore_index=True)
df_completo.to_csv('Emigrantes_TODOS_Con_Estudios.csv', index=False)

print(f"\n{'='*60}")
print(f"FICHERO COMPLETO: Emigrantes_TODOS_Con_Estudios.csv")
print(f"Total filas: {len(df_completo):,}")
print(f"Años procesados: {sorted(df_completo['ANIO'].unique().tolist())}")
print(f"{'='*60}")


PROCESANDO AÑO 2008
1. Cargando EVR_2008.csv...
   Total registros EVR: 2,635,679
   Emigrantes al extranjero: 266,460
2. Cargando EPA_2008T4.csv...
3. Realizando statistical matching...
4. Año 2008 completado: 256,638 emigrantes

PROCESANDO AÑO 2009
1. Cargando EVR_2009.csv...
   Total registros EVR: 2,475,632
   Emigrantes al extranjero: 323,641
2. Cargando EPA_2009T4.csv...
3. Realizando statistical matching...
4. Año 2009 completado: 313,317 emigrantes

PROCESANDO AÑO 2010
1. Cargando EVR_2010.csv...
   Total registros EVR: 2,519,792
   Emigrantes al extranjero: 373,954
2. Cargando EPA_2010T4.csv...
3. Realizando statistical matching...
4. Año 2010 completado: 361,839 emigrantes

PROCESANDO AÑO 2011
1. Cargando EVR_2011.csv...
   Total registros EVR: 2,475,524
   Emigrantes al extranjero: 370,540
2. Cargando EPA_2011T4.csv...
3. Realizando statistical matching...
4. Año 2011 completado: 358,208 emigrantes

PROCESANDO AÑO 2012
1. Cargando EVR_2012.csv...
   Total registros EVR: 2,3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

# =============================================================================
# AÑOS A PROCESAR
# =============================================================================
ANIOS = [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015,
         2016, 2017, 2018, 2019, 2020]

# =============================================================================
# FUNCIÓN AUXILIAR: agrupar edad
# =============================================================================
def agrupar_edad(edad):
    if pd.isna(edad): return '00'
    edad = int(edad)
    if edad < 16: return '00'
    elif edad <= 19: return '15'
    elif edad <= 24: return '20'
    elif edad <= 29: return '25'
    elif edad <= 34: return '30'
    elif edad <= 39: return '35'
    elif edad <= 44: return '40'
    elif edad <= 49: return '45'
    elif edad <= 54: return '50'
    elif edad <= 59: return '55'
    elif edad <= 64: return '60'
    else: return '65'

# =============================================================================
# FUNCIÓN PRINCIPAL: procesar un año
# =============================================================================
def procesar_anio(anio):
    print(f"\n{'='*60}")
    print(f"PROCESANDO AÑO {anio}")
    print(f"{'='*60}")

    # 1. Cargar EVR
    print(f"1. Cargando EVR_{anio}.csv...")
    df_evr = pd.read_csv(f'EVR_{anio}.csv', sep='\t', low_memory=False)
    print(f"   Total registros EVR: {len(df_evr):,}")

    df_evr['PROVALTA'] = df_evr['PROVALTA'].astype(str).str.zfill(2)
    df_emigrantes = df_evr[df_evr['PROVALTA'] == '66'].copy()
    print(f"   Emigrantes al extranjero: {len(df_emigrantes):,}")

    df_emigrantes['LLAVE_EDAD'] = df_emigrantes['EDAD'].apply(agrupar_edad)
    df_emigrantes['LLAVE_NAC'] = np.where(df_emigrantes['CNAC'].astype(str) == '108', '1', '2')

    # 2. Cargar EPA T4 (modo robusto: ignora comillas y líneas problemáticas)
    print(f"2. Cargando EPA_{anio}T4.csv (modo robusto)...")
    df_epa = pd.read_csv(
        f'EPA_{anio}T4.csv',
        sep='\t',
        quoting=3,
        on_bad_lines='skip',
        low_memory=False
    )
    print(f"   Filas EPA cargadas: {len(df_epa):,}")
    col_peso = df_epa.columns[-1]

    # 3. Crear matriz probabilística
    df_epa['LLAVE_EDAD'] = df_epa['EDAD5'].astype(str).str.zfill(2)
    df_epa['LLAVE_NAC'] = np.where(df_epa['NAC1'].astype(str) == '1', '1', '2')

    columnas_clave = ['LLAVE_NAC', 'LLAVE_EDAD']
    df_epa_valida = df_epa[df_epa['NFORMA'].notna()].copy()

    epa_agrupada = df_epa_valida.groupby(columnas_clave + ['NFORMA'])[col_peso].sum().reset_index()
    epa_totales = epa_agrupada.groupby(columnas_clave)[col_peso].sum().reset_index()
    epa_totales.rename(columns={col_peso: 'total'}, inplace=True)

    epa_probs = pd.merge(epa_agrupada, epa_totales, on=columnas_clave)
    epa_probs['prob'] = epa_probs[col_peso] / epa_probs['total']
    epa_matriz = epa_probs.pivot(index=columnas_clave, columns='NFORMA', values='prob').fillna(0).reset_index()

    # 4. Statistical matching
    print(f"3. Realizando statistical matching...")
    df_final = pd.merge(df_emigrantes, epa_matriz, on=columnas_clave, how='inner')

    niveles = [col for col in epa_matriz.columns if col not in columnas_clave]

    def asignar_estudios(fila):
        probs = [fila[n] for n in niveles]
        if sum(probs) == 0: return np.nan
        probs = probs / np.sum(probs)
        return np.random.choice(niveles, p=probs)

    df_final['NFORMA_IMPUTADO'] = df_final.apply(asignar_estudios, axis=1)
    df_final['ANIO'] = anio

    df_final.drop(columns=niveles + ['LLAVE_EDAD', 'LLAVE_NAC'], inplace=True)

    print(f"4. Año {anio} completado: {len(df_final):,} emigrantes")

    return df_final

# =============================================================================
# EJECUTAR PARA TODOS LOS AÑOS
# =============================================================================
np.random.seed(42)

dfs_todos = []
for anio in ANIOS:
    try:
        df = procesar_anio(anio)
        dfs_todos.append(df)
    except FileNotFoundError as e:
        print(f"\n¡AVISO! Falta archivo del año {anio}: {e}")
        continue
    except Exception as e:
        print(f"\n¡ERROR en año {anio}!: {e}")
        continue

# Concatenamos todos los años
df_completo = pd.concat(dfs_todos, ignore_index=True)
df_completo.to_csv('Emigrantes_TODOS_Con_Estudios.csv', index=False)

print(f"\n{'='*60}")
print(f"FICHERO COMPLETO: Emigrantes_TODOS_Con_Estudios.csv")
print(f"Total filas: {len(df_completo):,}")
print(f"Años procesados: {sorted(df_completo['ANIO'].unique().tolist())}")
print(f"{'='*60}")


PROCESANDO AÑO 2008
1. Cargando EVR_2008.csv...

¡AVISO! Falta archivo del año 2008: [Errno 2] No such file or directory: 'EVR_2008.csv'

PROCESANDO AÑO 2009
1. Cargando EVR_2009.csv...

¡AVISO! Falta archivo del año 2009: [Errno 2] No such file or directory: 'EVR_2009.csv'

PROCESANDO AÑO 2010
1. Cargando EVR_2010.csv...

¡AVISO! Falta archivo del año 2010: [Errno 2] No such file or directory: 'EVR_2010.csv'

PROCESANDO AÑO 2011
1. Cargando EVR_2011.csv...

¡AVISO! Falta archivo del año 2011: [Errno 2] No such file or directory: 'EVR_2011.csv'

PROCESANDO AÑO 2012
1. Cargando EVR_2012.csv...

¡AVISO! Falta archivo del año 2012: [Errno 2] No such file or directory: 'EVR_2012.csv'

PROCESANDO AÑO 2013
1. Cargando EVR_2013.csv...

¡AVISO! Falta archivo del año 2013: [Errno 2] No such file or directory: 'EVR_2013.csv'

PROCESANDO AÑO 2014
1. Cargando EVR_2014.csv...

¡AVISO! Falta archivo del año 2014: [Errno 2] No such file or directory: 'EVR_2014.csv'

PROCESANDO AÑO 2015
1. Cargando E

ValueError: No objects to concatenate

In [ ]:
import pandas as pd
import numpy as np

ANIOS = [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015,
         2016, 2017, 2018, 2019, 2020]

def agrupar_edad(edad):
    if pd.isna(edad): return '00'
    edad = int(edad)
    if edad < 16: return '00'
    elif edad <= 19: return '15'
    elif edad <= 24: return '20'
    elif edad <= 29: return '25'
    elif edad <= 34: return '30'
    elif edad <= 39: return '35'
    elif edad <= 44: return '40'
    elif edad <= 49: return '45'
    elif edad <= 54: return '50'
    elif edad <= 59: return '55'
    elif edad <= 64: return '60'
    else: return '65'

def cargar_epa_robusto(anio):
    """Intenta cargar la EPA con varias estrategias hasta que funcione."""
    archivo = f'EPA_{anio}T4.csv'

    # Estrategia 1: lectura normal
    try:
        df = pd.read_csv(archivo, sep='\t', low_memory=False)
        if 'EDAD5' in df.columns:
            return df
    except:
        pass

    # Estrategia 2: ignorar líneas problemáticas (mantiene comillas)
    try:
        df = pd.read_csv(archivo, sep='\t', on_bad_lines='skip', low_memory=False)
        if 'EDAD5' in df.columns:
            return df
    except:
        pass

    # Estrategia 3: motor de python (más lento pero más tolerante)
    try:
        df = pd.read_csv(archivo, sep='\t', engine='python', on_bad_lines='skip')
        if 'EDAD5' in df.columns:
            return df
    except:
        pass

    # Estrategia 4: ignorar comillas
    df = pd.read_csv(archivo, sep='\t', quoting=3, on_bad_lines='skip',
                     engine='python', low_memory=False)
    return df


def procesar_anio(anio):
    print(f"\n{'='*60}")
    print(f"PROCESANDO AÑO {anio}")
    print(f"{'='*60}")

    print(f"1. Cargando EVR_{anio}.csv...")
    df_evr = pd.read_csv(f'EVR_{anio}.csv', sep='\t', low_memory=False)
    print(f"   Total registros EVR: {len(df_evr):,}")

    df_evr['PROVALTA'] = df_evr['PROVALTA'].astype(str).str.zfill(2)
    df_emigrantes = df_evr[df_evr['PROVALTA'] == '66'].copy()
    print(f"   Emigrantes al extranjero: {len(df_emigrantes):,}")

    df_emigrantes['LLAVE_EDAD'] = df_emigrantes['EDAD'].apply(agrupar_edad)
    df_emigrantes['LLAVE_NAC'] = np.where(df_emigrantes['CNAC'].astype(str) == '108', '1', '2')

    print(f"2. Cargando EPA_{anio}T4.csv...")
    df_epa = cargar_epa_robusto(anio)
    print(f"   Filas EPA cargadas: {len(df_epa):,}")
    print(f"   Columnas detectadas: {list(df_epa.columns)[:8]}...")

    if 'EDAD5' not in df_epa.columns:
        raise ValueError(f"No se pudo encontrar EDAD5 en EPA_{anio}T4. Columnas: {list(df_epa.columns)}")

    col_peso = df_epa.columns[-1]

    df_epa['LLAVE_EDAD'] = df_epa['EDAD5'].astype(str).str.zfill(2)
    df_epa['LLAVE_NAC'] = np.where(df_epa['NAC1'].astype(str) == '1', '1', '2')

    columnas_clave = ['LLAVE_NAC', 'LLAVE_EDAD']
    df_epa_valida = df_epa[df_epa['NFORMA'].notna()].copy()

    epa_agrupada = df_epa_valida.groupby(columnas_clave + ['NFORMA'])[col_peso].sum().reset_index()
    epa_totales = epa_agrupada.groupby(columnas_clave)[col_peso].sum().reset_index()
    epa_totales.rename(columns={col_peso: 'total'}, inplace=True)

    epa_probs = pd.merge(epa_agrupada, epa_totales, on=columnas_clave)
    epa_probs['prob'] = epa_probs[col_peso] / epa_probs['total']
    epa_matriz = epa_probs.pivot(index=columnas_clave, columns='NFORMA', values='prob').fillna(0).reset_index()

    print(f"3. Realizando statistical matching...")
    df_final = pd.merge(df_emigrantes, epa_matriz, on=columnas_clave, how='inner')

    niveles = [col for col in epa_matriz.columns if col not in columnas_clave]

    def asignar_estudios(fila):
        probs = [fila[n] for n in niveles]
        if sum(probs) == 0: return np.nan
        probs = probs / np.sum(probs)
        return np.random.choice(niveles, p=probs)

    df_final['NFORMA_IMPUTADO'] = df_final.apply(asignar_estudios, axis=1)
    df_final['ANIO'] = anio

    df_final.drop(columns=niveles + ['LLAVE_EDAD', 'LLAVE_NAC'], inplace=True)

    print(f"4. Año {anio} completado: {len(df_final):,} emigrantes")

    return df_final

np.random.seed(42)

dfs_todos = []
for anio in ANIOS:
    try:
        df = procesar_anio(anio)
        dfs_todos.append(df)
    except FileNotFoundError as e:
        print(f"\n¡AVISO! Falta archivo del año {anio}: {e}")
        continue
    except Exception as e:
        print(f"\n¡ERROR en año {anio}!: {e}")
        continue

if len(dfs_todos) > 0:
    df_completo = pd.concat(dfs_todos, ignore_index=True)
    df_completo.to_csv('Emigrantes_TODOS_Con_Estudios.csv', index=False)

    print(f"\n{'='*60}")
    print(f"FICHERO COMPLETO: Emigrantes_TODOS_Con_Estudios.csv")
    print(f"Total filas: {len(df_completo):,}")
    print(f"Años procesados: {sorted(df_completo['ANIO'].unique().tolist())}")
    print(f"{'='*60}")
else:
    print("\nNo se procesó ningún año correctamente.")


PROCESANDO AÑO 2008
1. Cargando EVR_2008.csv...
   Total registros EVR: 2,635,679
   Emigrantes al extranjero: 266,460
2. Cargando EPA_2008T4.csv...
   Filas EPA cargadas: 166,699
   Columnas detectadas: ['CICLO', 'CCAA', 'PROV', 'NVIVI', 'NIVEL', 'NPERS', 'EDAD5', 'RELPP1']...
3. Realizando statistical matching...
4. Año 2008 completado: 256,638 emigrantes

PROCESANDO AÑO 2009
1. Cargando EVR_2009.csv...
   Total registros EVR: 2,475,632
   Emigrantes al extranjero: 323,641
2. Cargando EPA_2009T4.csv...
   Filas EPA cargadas: 175,682
   Columnas detectadas: ['CICLO', 'CCAA', 'PROV', 'NVIVI', 'NIVEL', 'NPERS', 'EDAD5', 'RELPP1']...
3. Realizando statistical matching...
4. Año 2009 completado: 313,317 emigrantes

PROCESANDO AÑO 2010
1. Cargando EVR_2010.csv...
   Total registros EVR: 2,519,792
   Emigrantes al extranjero: 373,954
2. Cargando EPA_2010T4.csv...
   Filas EPA cargadas: 170,932
   Columnas detectadas: ['CICLO', 'CCAA', 'PROV', 'NVIVI', 'NIVEL', 'NPERS', 'EDAD5', 'RELPP1']..

In [ ]:
import pandas as pd
import numpy as np

# =============================================================================
# AÑOS A PROCESAR
# =============================================================================
ANIOS = [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015,
         2016, 2017, 2018, 2019, 2020]

# =============================================================================
# DICCIONARIO PROVINCIA → CCAA (códigos oficiales del INE)
# =============================================================================
PROV_TO_CCAA = {
    1: 16, 2: 8, 3: 10, 4: 1, 5: 7, 6: 11, 7: 4, 8: 9, 9: 7, 10: 11,
    11: 1, 12: 10, 13: 8, 14: 1, 15: 12, 16: 8, 17: 9, 18: 1, 19: 8,
    20: 16, 21: 1, 22: 2, 23: 1, 24: 7, 25: 9, 26: 17, 27: 12, 28: 13,
    29: 1, 30: 14, 31: 15, 32: 12, 33: 3, 34: 7, 35: 5, 36: 12, 37: 7,
    38: 5, 39: 6, 40: 7, 41: 1, 42: 7, 43: 9, 44: 2, 45: 8, 46: 10,
    47: 7, 48: 16, 49: 7, 50: 2,
    51: 18, 52: 19  # Ceuta y Melilla
}

# =============================================================================
# FUNCIÓN AUXILIAR: agrupar edad
# =============================================================================
def agrupar_edad(edad):
    if pd.isna(edad): return '00'
    edad = int(edad)
    if edad < 16: return '00'
    elif edad <= 19: return '15'
    elif edad <= 24: return '20'
    elif edad <= 29: return '25'
    elif edad <= 34: return '30'
    elif edad <= 39: return '35'
    elif edad <= 44: return '40'
    elif edad <= 49: return '45'
    elif edad <= 54: return '50'
    elif edad <= 59: return '55'
    elif edad <= 64: return '60'
    else: return '65'

def cargar_epa_robusto(anio):
    archivo = f'EPA_{anio}T4.csv'
    try:
        df = pd.read_csv(archivo, sep='\t', low_memory=False)
        if 'EDAD5' in df.columns: return df
    except: pass
    try:
        df = pd.read_csv(archivo, sep='\t', on_bad_lines='skip', low_memory=False)
        if 'EDAD5' in df.columns: return df
    except: pass
    try:
        df = pd.read_csv(archivo, sep='\t', engine='python', on_bad_lines='skip')
        if 'EDAD5' in df.columns: return df
    except: pass
    df = pd.read_csv(archivo, sep='\t', quoting=3, on_bad_lines='skip',
                     engine='python', low_memory=False)
    return df

def procesar_anio(anio):
    print(f"\n{'='*60}")
    print(f"PROCESANDO AÑO {anio}")
    print(f"{'='*60}")

    print(f"1. Cargando EVR_{anio}.csv...")
    df_evr = pd.read_csv(f'EVR_{anio}.csv', sep='\t', low_memory=False)
    print(f"   Total registros EVR: {len(df_evr):,}")

    df_evr['PROVALTA'] = df_evr['PROVALTA'].astype(str).str.zfill(2)
    df_emigrantes = df_evr[df_evr['PROVALTA'] == '66'].copy()
    print(f"   Emigrantes al extranjero: {len(df_emigrantes):,}")

    # Crear llaves: edad, nacionalidad, sexo, CCAA
    df_emigrantes['LLAVE_EDAD'] = df_emigrantes['EDAD'].apply(agrupar_edad)
    df_emigrantes['LLAVE_NAC'] = np.where(df_emigrantes['CNAC'].astype(str) == '108', '1', '2')
    df_emigrantes['LLAVE_SEXO'] = df_emigrantes['SEXO'].astype(str)

    df_emigrantes['PROVBAJA_INT'] = pd.to_numeric(df_emigrantes['PROVBAJA'], errors='coerce')
    df_emigrantes['LLAVE_CCAA'] = df_emigrantes['PROVBAJA_INT'].map(PROV_TO_CCAA)
    df_emigrantes['LLAVE_CCAA'] = df_emigrantes['LLAVE_CCAA'].astype('Int64').astype(str)

    n_antes = len(df_emigrantes)
    df_emigrantes = df_emigrantes[df_emigrantes['LLAVE_CCAA'] != '<NA>'].copy()
    print(f"   Emigrantes con CCAA válida: {len(df_emigrantes):,} ({n_antes - len(df_emigrantes):,} descartados)")

    print(f"2. Cargando EPA_{anio}T4.csv...")
    df_epa = cargar_epa_robusto(anio)
    print(f"   Filas EPA cargadas: {len(df_epa):,}")

    if 'EDAD5' not in df_epa.columns:
        raise ValueError(f"No se pudo encontrar EDAD5. Columnas: {list(df_epa.columns)}")

    col_peso = df_epa.columns[-1]

    df_epa['LLAVE_EDAD'] = df_epa['EDAD5'].astype(str).str.zfill(2)
    df_epa['LLAVE_NAC'] = np.where(df_epa['NAC1'].astype(str) == '1', '1', '2')
    df_epa['LLAVE_SEXO'] = df_epa['SEXO1'].astype(str)
    df_epa['LLAVE_CCAA'] = pd.to_numeric(df_epa['CCAA'], errors='coerce').astype('Int64').astype(str)

    columnas_clave = ['LLAVE_NAC', 'LLAVE_EDAD', 'LLAVE_SEXO', 'LLAVE_CCAA']
    df_epa_valida = df_epa[df_epa['NFORMA'].notna()].copy()

    epa_agrupada = df_epa_valida.groupby(columnas_clave + ['NFORMA'])[col_peso].sum().reset_index()
    epa_totales = epa_agrupada.groupby(columnas_clave)[col_peso].sum().reset_index()
    epa_totales.rename(columns={col_peso: 'total'}, inplace=True)

    epa_probs = pd.merge(epa_agrupada, epa_totales, on=columnas_clave)
    epa_probs['prob'] = epa_probs[col_peso] / epa_probs['total']
    epa_matriz = epa_probs.pivot(index=columnas_clave, columns='NFORMA', values='prob').fillna(0).reset_index()

    print(f"3. Realizando statistical matching (4 llaves)...")
    df_final = pd.merge(df_emigrantes, epa_matriz, on=columnas_clave, how='inner')
    print(f"   Emparejamientos exitosos: {len(df_final):,} de {len(df_emigrantes):,}")

    niveles = [col for col in epa_matriz.columns if col not in columnas_clave]

    def asignar_estudios(fila):
        probs = [fila[n] for n in niveles]
        if sum(probs) == 0: return np.nan
        probs = probs / np.sum(probs)
        return np.random.choice(niveles, p=probs)

    df_final['NFORMA_IMPUTADO'] = df_final.apply(asignar_estudios, axis=1)
    df_final['ANIO'] = anio

    df_final.drop(columns=niveles + ['LLAVE_EDAD', 'LLAVE_NAC', 'LLAVE_SEXO',
                                      'LLAVE_CCAA', 'PROVBAJA_INT'], inplace=True)

    print(f"4. Año {anio} completado: {len(df_final):,} emigrantes")

    return df_final

np.random.seed(42)

dfs_todos = []
for anio in ANIOS:
    try:
        df = procesar_anio(anio)
        dfs_todos.append(df)
    except FileNotFoundError as e:
        print(f"\n¡AVISO! Falta archivo del año {anio}: {e}")
        continue
    except Exception as e:
        print(f"\n¡ERROR en año {anio}!: {e}")
        continue

if len(dfs_todos) > 0:
    df_completo = pd.concat(dfs_todos, ignore_index=True)
    df_completo.to_csv('Emigrantes_TODOS_Con_Estudios_v2.csv', index=False)

    print(f"\n{'='*60}")
    print(f"FICHERO COMPLETO v2: Emigrantes_TODOS_Con_Estudios_v2.csv")
    print(f"Total filas: {len(df_completo):,}")
    print(f"Años: {sorted(df_completo['ANIO'].unique().tolist())}")
    print(f"{'='*60}")
else:
    print("\nNo se procesó ningún año correctamente.")


PROCESANDO AÑO 2008
1. Cargando EVR_2008.csv...

¡ERROR en año 2008!: Error tokenizing data. C error: EOF inside string starting at row 2664279

PROCESANDO AÑO 2009
1. Cargando EVR_2009.csv...


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np

ANIOS = [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015,
         2016, 2017, 2018, 2019, 2020]

PROV_TO_CCAA = {
    1: 16, 2: 8, 3: 10, 4: 1, 5: 7, 6: 11, 7: 4, 8: 9, 9: 7, 10: 11,
    11: 1, 12: 10, 13: 8, 14: 1, 15: 12, 16: 8, 17: 9, 18: 1, 19: 8,
    20: 16, 21: 1, 22: 2, 23: 1, 24: 7, 25: 9, 26: 17, 27: 12, 28: 13,
    29: 1, 30: 14, 31: 15, 32: 12, 33: 3, 34: 7, 35: 5, 36: 12, 37: 7,
    38: 5, 39: 6, 40: 7, 41: 1, 42: 7, 43: 9, 44: 2, 45: 8, 46: 10,
    47: 7, 48: 16, 49: 7, 50: 2,
    51: 18, 52: 19
}

def agrupar_edad(edad):
    if pd.isna(edad): return '00'
    edad = int(edad)
    if edad < 16: return '00'
    elif edad <= 19: return '15'
    elif edad <= 24: return '20'
    elif edad <= 29: return '25'
    elif edad <= 34: return '30'
    elif edad <= 39: return '35'
    elif edad <= 44: return '40'
    elif edad <= 49: return '45'
    elif edad <= 54: return '50'
    elif edad <= 59: return '55'
    elif edad <= 64: return '60'
    else: return '65'

def cargar_robusto(archivo, validar_columna):
    """Carga un CSV intentando varias estrategias hasta que funcione."""
    try:
        df = pd.read_csv(archivo, sep='\t', low_memory=False)
        if validar_columna in df.columns: return df
    except: pass
    try:
        df = pd.read_csv(archivo, sep='\t', on_bad_lines='skip', low_memory=False)
        if validar_columna in df.columns: return df
    except: pass
    try:
        df = pd.read_csv(archivo, sep='\t', engine='python', on_bad_lines='skip')
        if validar_columna in df.columns: return df
    except: pass
    df = pd.read_csv(archivo, sep='\t', quoting=3, on_bad_lines='skip',
                     engine='python', low_memory=False)
    return df

def procesar_anio(anio):
    print(f"\n{'='*60}")
    print(f"PROCESANDO AÑO {anio}")
    print(f"{'='*60}")

    print(f"1. Cargando EVR_{anio}.csv...")
    df_evr = cargar_robusto(f'EVR_{anio}.csv', 'PROVALTA')
    print(f"   Total registros EVR: {len(df_evr):,}")

    df_evr['PROVALTA'] = df_evr['PROVALTA'].astype(str).str.zfill(2)
    df_emigrantes = df_evr[df_evr['PROVALTA'] == '66'].copy()
    print(f"   Emigrantes al extranjero: {len(df_emigrantes):,}")

    df_emigrantes['LLAVE_EDAD'] = df_emigrantes['EDAD'].apply(agrupar_edad)
    df_emigrantes['LLAVE_NAC'] = np.where(df_emigrantes['CNAC'].astype(str) == '108', '1', '2')
    df_emigrantes['LLAVE_SEXO'] = df_emigrantes['SEXO'].astype(str)

    df_emigrantes['PROVBAJA_INT'] = pd.to_numeric(df_emigrantes['PROVBAJA'], errors='coerce')
    df_emigrantes['LLAVE_CCAA'] = df_emigrantes['PROVBAJA_INT'].map(PROV_TO_CCAA)
    df_emigrantes['LLAVE_CCAA'] = df_emigrantes['LLAVE_CCAA'].astype('Int64').astype(str)

    n_antes = len(df_emigrantes)
    df_emigrantes = df_emigrantes[df_emigrantes['LLAVE_CCAA'] != '<NA>'].copy()
    print(f"   Emigrantes con CCAA válida: {len(df_emigrantes):,} ({n_antes - len(df_emigrantes):,} descartados)")

    print(f"2. Cargando EPA_{anio}T4.csv...")
    df_epa = cargar_robusto(f'EPA_{anio}T4.csv', 'EDAD5')
    print(f"   Filas EPA cargadas: {len(df_epa):,}")

    if 'EDAD5' not in df_epa.columns:
        raise ValueError(f"No se pudo encontrar EDAD5. Columnas: {list(df_epa.columns)}")

    col_peso = df_epa.columns[-1]

    df_epa['LLAVE_EDAD'] = df_epa['EDAD5'].astype(str).str.zfill(2)
    df_epa['LLAVE_NAC'] = np.where(df_epa['NAC1'].astype(str) == '1', '1', '2')
    df_epa['LLAVE_SEXO'] = df_epa['SEXO1'].astype(str)
    df_epa['LLAVE_CCAA'] = pd.to_numeric(df_epa['CCAA'], errors='coerce').astype('Int64').astype(str)

    columnas_clave = ['LLAVE_NAC', 'LLAVE_EDAD', 'LLAVE_SEXO', 'LLAVE_CCAA']
    df_epa_valida = df_epa[df_epa['NFORMA'].notna()].copy()

    epa_agrupada = df_epa_valida.groupby(columnas_clave + ['NFORMA'])[col_peso].sum().reset_index()
    epa_totales = epa_agrupada.groupby(columnas_clave)[col_peso].sum().reset_index()
    epa_totales.rename(columns={col_peso: 'total'}, inplace=True)

    epa_probs = pd.merge(epa_agrupada, epa_totales, on=columnas_clave)
    epa_probs['prob'] = epa_probs[col_peso] / epa_probs['total']
    epa_matriz = epa_probs.pivot(index=columnas_clave, columns='NFORMA', values='prob').fillna(0).reset_index()

    print(f"3. Realizando statistical matching (4 llaves)...")
    df_final = pd.merge(df_emigrantes, epa_matriz, on=columnas_clave, how='inner')
    print(f"   Emparejamientos exitosos: {len(df_final):,} de {len(df_emigrantes):,}")

    niveles = [col for col in epa_matriz.columns if col not in columnas_clave]

    def asignar_estudios(fila):
        probs = [fila[n] for n in niveles]
        if sum(probs) == 0: return np.nan
        probs = probs / np.sum(probs)
        return np.random.choice(niveles, p=probs)

    df_final['NFORMA_IMPUTADO'] = df_final.apply(asignar_estudios, axis=1)
    df_final['ANIO'] = anio

    df_final.drop(columns=niveles + ['LLAVE_EDAD', 'LLAVE_NAC', 'LLAVE_SEXO',
                                      'LLAVE_CCAA', 'PROVBAJA_INT'], inplace=True)

    print(f"4. Año {anio} completado: {len(df_final):,} emigrantes")

    return df_final

np.random.seed(42)

dfs_todos = []
for anio in ANIOS:
    try:
        df = procesar_anio(anio)
        dfs_todos.append(df)
    except FileNotFoundError as e:
        print(f"\n¡AVISO! Falta archivo del año {anio}: {e}")
        continue
    except Exception as e:
        print(f"\n¡ERROR en año {anio}!: {e}")
        continue

if len(dfs_todos) > 0:
    df_completo = pd.concat(dfs_todos, ignore_index=True)
    df_completo.to_csv('Emigrantes_TODOS_Con_Estudios_v2.csv', index=False)

    print(f"\n{'='*60}")
    print(f"FICHERO COMPLETO v2: Emigrantes_TODOS_Con_Estudios_v2.csv")
    print(f"Total filas: {len(df_completo):,}")
    print(f"Años: {sorted(df_completo['ANIO'].unique().tolist())}")
    print(f"{'='*60}")
else:
    print("\nNo se procesó ningún año correctamente.")


PROCESANDO AÑO 2008
1. Cargando EVR_2008.csv...
   Total registros EVR: 2,664,277
   Emigrantes al extranjero: 267,001
   Emigrantes con CCAA válida: 267,001 (0 descartados)
2. Cargando EPA_2008T4.csv...
   Filas EPA cargadas: 166,699
3. Realizando statistical matching (4 llaves)...
   Emparejamientos exitosos: 256,690 de 267,001
4. Año 2008 completado: 256,690 emigrantes

PROCESANDO AÑO 2009
1. Cargando EVR_2009.csv...
   Total registros EVR: 2,475,632
   Emigrantes al extranjero: 323,641
   Emigrantes con CCAA válida: 323,641 (0 descartados)
2. Cargando EPA_2009T4.csv...
   Filas EPA cargadas: 175,682
3. Realizando statistical matching (4 llaves)...
   Emparejamientos exitosos: 312,782 de 323,641
4. Año 2009 completado: 312,782 emigrantes

PROCESANDO AÑO 2010
1. Cargando EVR_2010.csv...
   Total registros EVR: 2,548,376
   Emigrantes al extranjero: 0
   Emigrantes con CCAA válida: 0 (0 descartados)
2. Cargando EPA_2010T4.csv...
   Filas EPA cargadas: 170,932
3. Realizando statistica

KeyboardInterrupt: 

In [ ]:
import pandas as pd

# Probamos las distintas estrategias para ver cuál se está usando
print("=== Estrategia 1: lectura normal ===")
try:
    df = pd.read_csv('EVR_2010.csv', sep='\t', low_memory=False)
    print(f"  OK. Filas: {len(df):,}, columnas: {list(df.columns)[:8]}")
    print(f"  PROVALTA primeros 5: {df['PROVALTA'].head().tolist()}")
    print(f"  Distinct PROVALTA: {sorted(df['PROVALTA'].unique())[:20]}")
except Exception as e:
    print(f"  FALLÓ: {e}")

print("\n=== Estrategia 2: on_bad_lines='skip' ===")
try:
    df = pd.read_csv('EVR_2010.csv', sep='\t', on_bad_lines='skip', low_memory=False)
    print(f"  OK. Filas: {len(df):,}, columnas: {list(df.columns)[:8]}")
    print(f"  PROVALTA primeros 5: {df['PROVALTA'].head().tolist()}")
except Exception as e:
    print(f"  FALLÓ: {e}")

=== Estrategia 1: lectura normal ===
  OK. Filas: 2,548,376, columnas: ['SEXO', 'PROVNAC', 'MUNINAC', 'EDAD', 'MESNAC', 'ANONAC', 'CNAC', 'PROVALTA']
  PROVALTA primeros 5: [1.0, 1.0, 20.0, 3.0, 1.0]
  Distinct PROVALTA: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0)]

=== Estrategia 2: on_bad_lines='skip' ===
  OK. Filas: 2,548,376, columnas: ['SEXO', 'PROVNAC', 'MUNINAC', 'EDAD', 'MESNAC', 'ANONAC', 'CNAC', 'PROVALTA']
  PROVALTA primeros 5: [1.0, 1.0, 20.0, 3.0, 1.0]


In [ ]:
import pandas as pd
import numpy as np

ANIOS = [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015,
         2016, 2017, 2018, 2019, 2020]

PROV_TO_CCAA = {
    1: 16, 2: 8, 3: 10, 4: 1, 5: 7, 6: 11, 7: 4, 8: 9, 9: 7, 10: 11,
    11: 1, 12: 10, 13: 8, 14: 1, 15: 12, 16: 8, 17: 9, 18: 1, 19: 8,
    20: 16, 21: 1, 22: 2, 23: 1, 24: 7, 25: 9, 26: 17, 27: 12, 28: 13,
    29: 1, 30: 14, 31: 15, 32: 12, 33: 3, 34: 7, 35: 5, 36: 12, 37: 7,
    38: 5, 39: 6, 40: 7, 41: 1, 42: 7, 43: 9, 44: 2, 45: 8, 46: 10,
    47: 7, 48: 16, 49: 7, 50: 2,
    51: 18, 52: 19
}

def agrupar_edad(edad):
    if pd.isna(edad): return '00'
    edad = int(edad)
    if edad < 16: return '00'
    elif edad <= 19: return '15'
    elif edad <= 24: return '20'
    elif edad <= 29: return '25'
    elif edad <= 34: return '30'
    elif edad <= 39: return '35'
    elif edad <= 44: return '40'
    elif edad <= 49: return '45'
    elif edad <= 54: return '50'
    elif edad <= 59: return '55'
    elif edad <= 64: return '60'
    else: return '65'

def cargar_robusto(archivo, validar_columna):
    try:
        df = pd.read_csv(archivo, sep='\t', low_memory=False)
        if validar_columna in df.columns: return df
    except: pass
    try:
        df = pd.read_csv(archivo, sep='\t', on_bad_lines='skip', low_memory=False)
        if validar_columna in df.columns: return df
    except: pass
    try:
        df = pd.read_csv(archivo, sep='\t', engine='python', on_bad_lines='skip')
        if validar_columna in df.columns: return df
    except: pass
    df = pd.read_csv(archivo, sep='\t', quoting=3, on_bad_lines='skip',
                     engine='python', low_memory=False)
    return df

def procesar_anio(anio):
    print(f"\n{'='*60}")
    print(f"PROCESANDO AÑO {anio}")
    print(f"{'='*60}")

    print(f"1. Cargando EVR_{anio}.csv...")
    df_evr = cargar_robusto(f'EVR_{anio}.csv', 'PROVALTA')
    print(f"   Total registros EVR: {len(df_evr):,}")

    # CORREGIDO: convierte número (1.0) o texto ("01") a string limpio ("01")
    df_evr['PROVALTA'] = pd.to_numeric(df_evr['PROVALTA'], errors='coerce').fillna(0).astype(int).astype(str).str.zfill(2)
    df_emigrantes = df_evr[df_evr['PROVALTA'] == '66'].copy()
    print(f"   Emigrantes al extranjero: {len(df_emigrantes):,}")

    df_emigrantes['LLAVE_EDAD'] = df_emigrantes['EDAD'].apply(agrupar_edad)
    df_emigrantes['LLAVE_NAC'] = np.where(df_emigrantes['CNAC'].astype(str) == '108', '1', '2')
    df_emigrantes['LLAVE_SEXO'] = df_emigrantes['SEXO'].astype(str)

    df_emigrantes['PROVBAJA_INT'] = pd.to_numeric(df_emigrantes['PROVBAJA'], errors='coerce')
    df_emigrantes['LLAVE_CCAA'] = df_emigrantes['PROVBAJA_INT'].map(PROV_TO_CCAA)
    df_emigrantes['LLAVE_CCAA'] = df_emigrantes['LLAVE_CCAA'].astype('Int64').astype(str)

    n_antes = len(df_emigrantes)
    df_emigrantes = df_emigrantes[df_emigrantes['LLAVE_CCAA'] != '<NA>'].copy()
    print(f"   Emigrantes con CCAA válida: {len(df_emigrantes):,} ({n_antes - len(df_emigrantes):,} descartados)")

    print(f"2. Cargando EPA_{anio}T4.csv...")
    df_epa = cargar_robusto(f'EPA_{anio}T4.csv', 'EDAD5')
    print(f"   Filas EPA cargadas: {len(df_epa):,}")

    if 'EDAD5' not in df_epa.columns:
        raise ValueError(f"No se pudo encontrar EDAD5. Columnas: {list(df_epa.columns)}")

    col_peso = df_epa.columns[-1]

    df_epa['LLAVE_EDAD'] = df_epa['EDAD5'].astype(str).str.zfill(2)
    df_epa['LLAVE_NAC'] = np.where(df_epa['NAC1'].astype(str) == '1', '1', '2')
    df_epa['LLAVE_SEXO'] = df_epa['SEXO1'].astype(str)
    df_epa['LLAVE_CCAA'] = pd.to_numeric(df_epa['CCAA'], errors='coerce').astype('Int64').astype(str)

    columnas_clave = ['LLAVE_NAC', 'LLAVE_EDAD', 'LLAVE_SEXO', 'LLAVE_CCAA']
    df_epa_valida = df_epa[df_epa['NFORMA'].notna()].copy()

    epa_agrupada = df_epa_valida.groupby(columnas_clave + ['NFORMA'])[col_peso].sum().reset_index()
    epa_totales = epa_agrupada.groupby(columnas_clave)[col_peso].sum().reset_index()
    epa_totales.rename(columns={col_peso: 'total'}, inplace=True)

    epa_probs = pd.merge(epa_agrupada, epa_totales, on=columnas_clave)
    epa_probs['prob'] = epa_probs[col_peso] / epa_probs['total']
    epa_matriz = epa_probs.pivot(index=columnas_clave, columns='NFORMA', values='prob').fillna(0).reset_index()

    print(f"3. Realizando statistical matching (4 llaves)...")
    df_final = pd.merge(df_emigrantes, epa_matriz, on=columnas_clave, how='inner')
    print(f"   Emparejamientos exitosos: {len(df_final):,} de {len(df_emigrantes):,}")

    niveles = [col for col in epa_matriz.columns if col not in columnas_clave]

    def asignar_estudios(fila):
        probs = [fila[n] for n in niveles]
        if sum(probs) == 0: return np.nan
        probs = probs / np.sum(probs)
        return np.random.choice(niveles, p=probs)

    df_final['NFORMA_IMPUTADO'] = df_final.apply(asignar_estudios, axis=1)
    df_final['ANIO'] = anio

    df_final.drop(columns=niveles + ['LLAVE_EDAD', 'LLAVE_NAC', 'LLAVE_SEXO',
                                      'LLAVE_CCAA', 'PROVBAJA_INT'], inplace=True)

    print(f"4. Año {anio} completado: {len(df_final):,} emigrantes")

    return df_final

np.random.seed(42)

dfs_todos = []
for anio in ANIOS:
    try:
        df = procesar_anio(anio)
        dfs_todos.append(df)
    except FileNotFoundError as e:
        print(f"\n¡AVISO! Falta archivo del año {anio}: {e}")
        continue
    except Exception as e:
        print(f"\n¡ERROR en año {anio}!: {e}")
        continue

if len(dfs_todos) > 0:
    df_completo = pd.concat(dfs_todos, ignore_index=True)
    df_completo.to_csv('Emigrantes_TODOS_Con_Estudios_v2.csv', index=False)

    print(f"\n{'='*60}")
    print(f"FICHERO COMPLETO v2: Emigrantes_TODOS_Con_Estudios_v2.csv")
    print(f"Total filas: {len(df_completo):,}")
    print(f"Años: {sorted(df_completo['ANIO'].unique().tolist())}")
    print(f"{'='*60}")
else:
    print("\nNo se procesó ningún año correctamente.")


PROCESANDO AÑO 2008
1. Cargando EVR_2008.csv...
   Total registros EVR: 2,664,277
   Emigrantes al extranjero: 267,001
   Emigrantes con CCAA válida: 267,001 (0 descartados)
2. Cargando EPA_2008T4.csv...
   Filas EPA cargadas: 166,699
3. Realizando statistical matching (4 llaves)...
   Emparejamientos exitosos: 256,690 de 267,001
4. Año 2008 completado: 256,690 emigrantes

PROCESANDO AÑO 2009
1. Cargando EVR_2009.csv...
   Total registros EVR: 2,475,632
   Emigrantes al extranjero: 323,641
   Emigrantes con CCAA válida: 323,641 (0 descartados)
2. Cargando EPA_2009T4.csv...
   Filas EPA cargadas: 175,682
3. Realizando statistical matching (4 llaves)...
   Emparejamientos exitosos: 312,782 de 323,641
4. Año 2009 completado: 312,782 emigrantes

PROCESANDO AÑO 2010
1. Cargando EVR_2010.csv...
   Total registros EVR: 2,548,376
   Emigrantes al extranjero: 374,485
   Emigrantes con CCAA válida: 374,485 (0 descartados)
2. Cargando EPA_2010T4.csv...
   Filas EPA cargadas: 170,932
3. Realizand

In [ ]:
import pandas as pd

# Comparar lectura de EVR_2010 vs EVR_2012
for anio in [2010, 2011, 2012]:
    print(f"\n=== EVR {anio} ===")
    df = pd.read_csv(f'EVR_{anio}.csv', sep='\t', low_memory=False)

    df['PROVALTA'] = pd.to_numeric(df['PROVALTA'], errors='coerce').fillna(0).astype(int).astype(str).str.zfill(2)
    emig = df[df['PROVALTA'] == '66'].copy()

    # Españoles 25-34
    emig_e = emig[(emig['CNAC'].astype(str) == '108') &
                  (emig['EDAD'] >= 25) & (emig['EDAD'] <= 34)]

    print(f"  Total emigrantes: {len(emig):,}")
    print(f"  Españoles 25-34: {len(emig_e):,}")

    # ¿Hay datos raros?
    print(f"  CCAA = PROVBAJA value_counts top 5:")
    print(emig_e['PROVBAJA'].value_counts().head())


=== EVR 2010 ===
  Total emigrantes: 374,485
  Españoles 25-34: 0
  CCAA = PROVBAJA value_counts top 5:
Series([], Name: count, dtype: int64)

=== EVR 2011 ===


TypeError: '>=' not supported between instances of 'str' and 'int'

In [ ]:
import pandas as pd
import numpy as np

ANIOS = [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015,
         2016, 2017, 2018, 2019, 2020]

PROV_TO_CCAA = {
    1: 16, 2: 8, 3: 10, 4: 1, 5: 7, 6: 11, 7: 4, 8: 9, 9: 7, 10: 11,
    11: 1, 12: 10, 13: 8, 14: 1, 15: 12, 16: 8, 17: 9, 18: 1, 19: 8,
    20: 16, 21: 1, 22: 2, 23: 1, 24: 7, 25: 9, 26: 17, 27: 12, 28: 13,
    29: 1, 30: 14, 31: 15, 32: 12, 33: 3, 34: 7, 35: 5, 36: 12, 37: 7,
    38: 5, 39: 6, 40: 7, 41: 1, 42: 7, 43: 9, 44: 2, 45: 8, 46: 10,
    47: 7, 48: 16, 49: 7, 50: 2,
    51: 18, 52: 19
}

def agrupar_edad(edad):
    if pd.isna(edad): return '00'
    edad = int(edad)
    if edad < 16: return '00'
    elif edad <= 19: return '15'
    elif edad <= 24: return '20'
    elif edad <= 29: return '25'
    elif edad <= 34: return '30'
    elif edad <= 39: return '35'
    elif edad <= 44: return '40'
    elif edad <= 49: return '45'
    elif edad <= 54: return '50'
    elif edad <= 59: return '55'
    elif edad <= 64: return '60'
    else: return '65'

def cargar_robusto(archivo, validar_columna):
    try:
        df = pd.read_csv(archivo, sep='\t', low_memory=False)
        if validar_columna in df.columns: return df
    except: pass
    try:
        df = pd.read_csv(archivo, sep='\t', on_bad_lines='skip', low_memory=False)
        if validar_columna in df.columns: return df
    except: pass
    try:
        df = pd.read_csv(archivo, sep='\t', engine='python', on_bad_lines='skip')
        if validar_columna in df.columns: return df
    except: pass
    df = pd.read_csv(archivo, sep='\t', quoting=3, on_bad_lines='skip',
                     engine='python', low_memory=False)
    return df

def procesar_anio(anio):
    print(f"\n{'='*60}")
    print(f"PROCESANDO AÑO {anio}")
    print(f"{'='*60}")

    print(f"1. Cargando EVR_{anio}.csv...")
    df_evr = cargar_robusto(f'EVR_{anio}.csv', 'PROVALTA')
    print(f"   Total registros EVR: {len(df_evr):,}")

    # CONVERSIÓN ROBUSTA DE TODAS LAS COLUMNAS NUMÉRICAS DE LA EVR
    df_evr['PROVALTA'] = pd.to_numeric(df_evr['PROVALTA'], errors='coerce').fillna(0).astype(int).astype(str).str.zfill(2)
    df_evr['EDAD'] = pd.to_numeric(df_evr['EDAD'], errors='coerce')
    df_evr['CNAC'] = pd.to_numeric(df_evr['CNAC'], errors='coerce').fillna(0).astype(int)
    df_evr['SEXO'] = pd.to_numeric(df_evr['SEXO'], errors='coerce').fillna(0).astype(int)
    df_evr['PROVBAJA'] = pd.to_numeric(df_evr['PROVBAJA'], errors='coerce')

    df_emigrantes = df_evr[df_evr['PROVALTA'] == '66'].copy()
    df_emigrantes = df_emigrantes[df_emigrantes['EDAD'].notna()].copy()
    print(f"   Emigrantes al extranjero: {len(df_emigrantes):,}")

    df_emigrantes['LLAVE_EDAD'] = df_emigrantes['EDAD'].apply(agrupar_edad)
    df_emigrantes['LLAVE_NAC'] = np.where(df_emigrantes['CNAC'] == 108, '1', '2')
    df_emigrantes['LLAVE_SEXO'] = df_emigrantes['SEXO'].astype(str)

    df_emigrantes['LLAVE_CCAA'] = df_emigrantes['PROVBAJA'].map(PROV_TO_CCAA)
    df_emigrantes['LLAVE_CCAA'] = df_emigrantes['LLAVE_CCAA'].astype('Int64').astype(str)

    n_antes = len(df_emigrantes)
    df_emigrantes = df_emigrantes[df_emigrantes['LLAVE_CCAA'] != '<NA>'].copy()
    print(f"   Emigrantes con CCAA válida: {len(df_emigrantes):,} ({n_antes - len(df_emigrantes):,} descartados)")

    print(f"2. Cargando EPA_{anio}T4.csv...")
    df_epa = cargar_robusto(f'EPA_{anio}T4.csv', 'EDAD5')
    print(f"   Filas EPA cargadas: {len(df_epa):,}")

    if 'EDAD5' not in df_epa.columns:
        raise ValueError(f"No se pudo encontrar EDAD5. Columnas: {list(df_epa.columns)}")

    col_peso = df_epa.columns[-1]

    # CONVERSIÓN ROBUSTA EN LA EPA TAMBIÉN
    df_epa['LLAVE_EDAD'] = df_epa['EDAD5'].astype(str).str.zfill(2)
    df_epa['NAC1'] = pd.to_numeric(df_epa['NAC1'], errors='coerce').fillna(0).astype(int)
    df_epa['LLAVE_NAC'] = np.where(df_epa['NAC1'] == 1, '1', '2')
    df_epa['SEXO1'] = pd.to_numeric(df_epa['SEXO1'], errors='coerce').fillna(0).astype(int)
    df_epa['LLAVE_SEXO'] = df_epa['SEXO1'].astype(str)
    df_epa['LLAVE_CCAA'] = pd.to_numeric(df_epa['CCAA'], errors='coerce').astype('Int64').astype(str)

    columnas_clave = ['LLAVE_NAC', 'LLAVE_EDAD', 'LLAVE_SEXO', 'LLAVE_CCAA']
    df_epa_valida = df_epa[df_epa['NFORMA'].notna()].copy()

    epa_agrupada = df_epa_valida.groupby(columnas_clave + ['NFORMA'])[col_peso].sum().reset_index()
    epa_totales = epa_agrupada.groupby(columnas_clave)[col_peso].sum().reset_index()
    epa_totales.rename(columns={col_peso: 'total'}, inplace=True)

    epa_probs = pd.merge(epa_agrupada, epa_totales, on=columnas_clave)
    epa_probs['prob'] = epa_probs[col_peso] / epa_probs['total']
    epa_matriz = epa_probs.pivot(index=columnas_clave, columns='NFORMA', values='prob').fillna(0).reset_index()

    print(f"3. Realizando statistical matching (4 llaves)...")
    df_final = pd.merge(df_emigrantes, epa_matriz, on=columnas_clave, how='inner')
    print(f"   Emparejamientos exitosos: {len(df_final):,} de {len(df_emigrantes):,}")

    niveles = [col for col in epa_matriz.columns if col not in columnas_clave]

    def asignar_estudios(fila):
        probs = [fila[n] for n in niveles]
        if sum(probs) == 0: return np.nan
        probs = probs / np.sum(probs)
        return np.random.choice(niveles, p=probs)

    df_final['NFORMA_IMPUTADO'] = df_final.apply(asignar_estudios, axis=1)
    df_final['ANIO'] = anio

    df_final.drop(columns=niveles + ['LLAVE_EDAD', 'LLAVE_NAC', 'LLAVE_SEXO',
                                      'LLAVE_CCAA'], inplace=True)

    print(f"4. Año {anio} completado: {len(df_final):,} emigrantes")

    return df_final

np.random.seed(42)

dfs_todos = []
for anio in ANIOS:
    try:
        df = procesar_anio(anio)
        dfs_todos.append(df)
    except FileNotFoundError as e:
        print(f"\n¡AVISO! Falta archivo del año {anio}: {e}")
        continue
    except Exception as e:
        print(f"\n¡ERROR en año {anio}!: {e}")
        continue

if len(dfs_todos) > 0:
    df_completo = pd.concat(dfs_todos, ignore_index=True)
    df_completo.to_csv('Emigrantes_TODOS_Con_Estudios_v3.csv', index=False)

    print(f"\n{'='*60}")
    print(f"FICHERO COMPLETO v3: Emigrantes_TODOS_Con_Estudios_v3.csv")
    print(f"Total filas: {len(df_completo):,}")
    print(f"Años: {sorted(df_completo['ANIO'].unique().tolist())}")
    print(f"{'='*60}")
else:
    print("\nNo se procesó ningún año correctamente.")


PROCESANDO AÑO 2008
1. Cargando EVR_2008.csv...
   Total registros EVR: 2,664,277
   Emigrantes al extranjero: 267,001
   Emigrantes con CCAA válida: 267,001 (0 descartados)
2. Cargando EPA_2008T4.csv...
   Filas EPA cargadas: 166,699
3. Realizando statistical matching (4 llaves)...
   Emparejamientos exitosos: 256,690 de 267,001
4. Año 2008 completado: 256,690 emigrantes

PROCESANDO AÑO 2009
1. Cargando EVR_2009.csv...
   Total registros EVR: 2,475,632
   Emigrantes al extranjero: 323,641
   Emigrantes con CCAA válida: 323,641 (0 descartados)
2. Cargando EPA_2009T4.csv...
   Filas EPA cargadas: 175,682
3. Realizando statistical matching (4 llaves)...
   Emparejamientos exitosos: 312,782 de 323,641
4. Año 2009 completado: 312,782 emigrantes

PROCESANDO AÑO 2010
1. Cargando EVR_2010.csv...
   Total registros EVR: 2,548,376
   Emigrantes al extranjero: 374,485
   Emigrantes con CCAA válida: 374,485 (0 descartados)
2. Cargando EPA_2010T4.csv...
   Filas EPA cargadas: 170,932
3. Realizand